# Exploratory notebook only

This notebook is not the production source for Tables 3-5. The production task is `make_summary_stats.py`, run through `code/Makefile`. Do not use this notebook to regenerate paper artifacts.

# Table 3 Summary Statistics

Run from the task code directory or from the repository root. Edit the play-area variables, then rerun to inspect the source values and LaTeX table.

In [ ]:
from pathlib import Path
import pandas as pd
import yaml
from IPython.display import display

code_dir = Path.cwd()
if code_dir.name != "code":
    code_dir = Path("tasks/measurement/summary_stats/code").resolve()
task_dir = code_dir.parent
repo_dir = task_dir.parents[2]
input_dir = task_dir / "input"
output_dir = task_dir / "output" / "summarystats"

df = pd.read_stata(input_dir / "MMB_reg_format.dta")

with open(repo_dir / "config" / "params.yaml", "r", encoding="utf-8") as f:
    summary_time_limit = yaml.safe_load(f)["mmb"]["time_limit"]

# Play area: adjust these if you want to inspect variants.
table_variables = [
    ("IScurve20", "\\textit{y-slope}"),
    ("infl_per_rr20", "\\textit{$\\pi$-slope}"),
    ("sacratio20", "\\textit{sacratio}"),
    ("y_timing_max", "\\textit{y-timing}"),
    ("piq_timing_max", "\\textit{$\\pi$-timing}"),
]
row_end = "\\\\"

table_df = df.loc[df["y_timing_max"] < summary_time_limit].copy()
table_df = table_df.loc[table_df["piq_timing_max"] < summary_time_limit].copy()
table_groups = [
    ("Full sample", table_df),
    ("Calibrated models", table_df.loc[table_df["calibrated"] == 1].copy()),
    ("Estimated models", table_df.loc[table_df["estimated"] == 1].copy()),
]
table_stats = [("Mean", "mean"), ("Median", "median"), ("Std deviation", "sd"), ("Skewness", "skew"), ("N", "n")]

summary_rows = []
for group_name, group_df in table_groups:
    for var, column_label in table_variables:
        x = pd.to_numeric(group_df[var], errors="coerce").dropna()
        summary_rows.append({
            "group": group_name,
            "variable": var,
            "column_label": column_label,
            "mean": x.mean(),
            "median": x.median(),
            "sd": x.std(),
            "skew": x.skew(),
            "n": int(x.shape[0]),
        })

table_summary = pd.DataFrame(summary_rows)
display(table_summary)

latex_lines = [
    "\\begin{table}[H]",
    "\\centering",
    "\\caption{ ",
    "\\\\",
    "\\underline{Summary statistics for the constructed data}}",
    "\\label{tab:summary_stats}",
    "\\small",
    "\\begin{tabular}{l c c c c c}",
    "\\toprule",
]

for group_index, (group_name, group_df) in enumerate(table_groups):
    if group_index > 0:
        latex_lines.append("\\midrule")
    latex_lines.append(f"\\textbf{{{group_name}}} & & & & & " + row_end)
    latex_lines.append(" & " + " & ".join(label for _, label in table_variables) + " " + row_end)
    latex_lines.append("\\midrule")
    group_summary = table_summary.loc[table_summary["group"] == group_name].set_index("variable")
    for row_label, stat_name in table_stats:
        cells = []
        for var, _ in table_variables:
            value = group_summary.loc[var, stat_name]
            if stat_name == "n":
                cells.append(str(int(value)))
            else:
                text = f"{float(value):.2f}"
                if text == "-0.00":
                    text = "0.00"
                if float(value) < 0:
                    text = f"${text}$"
                cells.append(text)
        latex_lines.append(f"{row_label} & " + " & ".join(cells) + " " + row_end)

latex_lines.extend([
    "\\bottomrule",
    "\\multicolumn{6}{p{13cm}}{\\footnotesize Notes: See Table~\\ref{tab:outcome_vars} for definitions of variables. Source: authors' calculations based on simulations of MMB models.}",
    "\\end{tabular}",
    "\\end{table}",
])

table3_latex = "\n".join(latex_lines) + "\n"
print(table3_latex)

print("Production LaTeX output:", output_dir / "table3_summary_stats.tex")


## Table 4 elasticity statistics

This cell displays the production Table 4 source values and LaTeX output.

In [ ]:
table4_csv = output_dir / "table4_elasticity_stats.csv"
table4_tex = output_dir / "table4_elasticity_stats.tex"
table4_values = pd.read_csv(table4_csv)
display(table4_values)
print(table4_tex.read_text(encoding="utf-8"))


## Table 5 timing statistics

This cell displays the production Table 5 source values and LaTeX output.

In [ ]:
table5_csv = output_dir / "table5_timing_stats.csv"
table5_tex = output_dir / "table5_timing_stats.tex"
table5_values = pd.read_csv(table5_csv)
display(table5_values)
print(table5_tex.read_text(encoding="utf-8"))
